Validamos la tabla fuente

In [0]:
%sql
SELECT COUNT(*) AS total FROM bootcamp.silver.propiedades;

## 1.1: Crear dimensiones para Star Schema


In [0]:
%sql
-- Crear dim_Zona

DROP TABLE IF EXISTS propiedades.gold.dim_zona;

CREATE TABLE propiedades.gold.dim_zona
(
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    partido STRING NOT NULL,
    region  STRING NOT NULL,
    ciudad  STRING,
    provincia   STRING DEFAULT 'Buenos Aires',
    pais        STRING DEFAULT 'Argentina',
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimension de Zonas - Start Schema';

--------------------------------------------------------------
-- Insertamos Datos
--------------------------------------------------------------
INSERT INTO propiedades.gold.dim_zona
(partido, region, ciudad, provincia,pais)
SELECT DISTINCT
        partido,
        region,
        CASE 
            WHEN sp.region = 'capital federal' THEN 'CABA'
            ELSE 'GBA'
        END as ciudad,
        'Buenos Aires'  as provincia,
        'Argentina'     as pais

FROM    propiedades.silver.propiedades sp
WHERE   partido IS NOT NULL
ORDER BY partido
;


In [0]:
%sql
-- dim_tipo_operacion

DROP TABLE IF EXISTS propiedades.gold.dim_tipo_operacion;

CREATE TABLE propiedades.gold.dim_tipo_operacion
(
    tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    tipo_operacion STRING NOT NULL,
    moneda      STRING NOT NULL,
    categoria   STRING,
    descripcion STRING,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA 
TBLPROPERTIES(
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimension de tipo de operacion + moneda - Star Schema'
;


--------------------------------------------------------------
-- Insertamos Datos
--------------------------------------------------------------

INSERT INTO propiedades.gold.dim_tipo_operacion
(tipo_operacion, moneda, categoria, descripcion)
SELECT  DISTINCT
        tipo_operacion,
        moneda,
        CASE 
            WHEN tipo_operacion = 'alquiler' THEN 'residencial'
            WHEN tipo_operacion = 'venta' THEN 'residencial'
            WHEN tipo_operacion = 'alquiler_temporario' THEN 'temporal'
            ELSE 'otro'
        END as categoria,
        CASE 
            WHEN tipo_operacion = 'alquiler' THEN 'Alquiler residencial'
            WHEN tipo_operacion = 'venta' THEN 'Venta de propiedad'
            WHEN tipo_operacion = 'alquiler_temporario' THEN 'Alquiler temporal'
            ELSE 'Otro tipo'
        END as descripcion     
                   
FROM    propiedades.silver.propiedades
WHERE   tipo_operacion IS NOT NULL
AND     moneda is not NULL
ORDER BY tipo_operacion, moneda
;


In [0]:
%sql
-- dim_tiempo

DROP TABLE IF EXISTS propiedades.gold.dim_tiempo;

CREATE TABLE propiedades.gold.dim_tiempo (
    fecha_id    BIGINT,
    fecha       DATE NOT NULL,
    anio        INT,
    mes         INT,
    trimestre   INT,
    dia_semana  STRING,
    es_fin_de_semana BOOLEAN,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES(
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimension de tiempo - Star Schema'
;


---------------------------
-- Insertamos datos iniciales a la tabla
-----------------------------

INSERT INTO propiedades.gold.dim_tiempo
(fecha_id, fecha,ANIO,MES,trimestre,dia_semana,es_fin_de_semana)

SELECT DISTINCT
        CAST(DATE_FORMAT(fecha_publicacion, 'yyyyMMdd') AS BIGINT) as fecha_id,
        fecha_publicacion as fecha,
        YEAR(fecha_publicacion) as anio,
        MONTH(fecha_publicacion) as mes,
        QUARTER(fecha_publicacion) as trimestre,
        CASE DAYOFWEEK(fecha_publicacion)
            WHEN 1 THEN 'Domingo'
            WHEN 2 THEN 'Lunes'
            WHEN 3 THEN 'Martes'
            WHEN 4 THEN 'Miércoles'
            WHEN 5 THEN 'Jueves'
            WHEN 6 THEN 'Viernes'
            WHEN 7 THEN 'Sábado'
        END as dia_semana,
        DAYOFWEEK(fecha_publicacion) IN (1, 7) as es_fin_de_semana
FROM    propiedades.silver.propiedades
WHERE fecha_publicacion IS NOT NULL
;


In [0]:
%sql
-- dim_caracteristicas (Kimball junk dimension

DROP TABLE IF EXISTS propiedades.gold.dim_caracteristicas;

CREATE TABLE propiedades.gold.dim_caracteristicas
(
    caracteristicas_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    estado STRING NOT NULL,
    cochera BOOLEAN,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES
(
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimensión de características de la propiedad (estado + cochera) - Star Schema'
;

---------------------------------------------------------
-- INSERTAMOS DATOS INICIALES
---------------------------------------------------------
INSERT INTO propiedades.gold.dim_caracteristicas
(estado, cochera)
SELECT  DISTINCT
        coalesce(estado, 'sin especificar') AS estado,
        coalesce(cochera, false) as cochera

FROM    propiedades.silver.propiedades
ORDER BY estado,cochera


In [0]:
%sql
-- dim_orientacion

DROP TABLE IF EXISTS propiedades.gold.dim_orientacion;

CREATE TABLE propiedades.gold.dim_orientacion
(   orientacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    orientacion STRING NOT NULL,
    tipo_orientacion STRING,
    _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES
(
    'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT 'Dimensión de orientación - Star Schema'
;

---------------------------------------
-- Insertamos datos dim_orientacion
---------------------------------------

INSERT INTO propiedades.gold.dim_orientacion
(orientacion, tipo_orientacion)
SELECT  DISTINCT
        coalesce(orientacion, 'sin especificar') AS orientacion,
        CASE 
            WHEN LOWER(orientacion) LIKE '%norte%' THEN 'norte'
            WHEN LOWER(orientacion) LIKE '%sur%' THEN 'sur'
            WHEN LOWER(orientacion) LIKE '%este%' THEN 'este'
            WHEN LOWER(orientacion) LIKE '%oeste%' THEN 'oeste'
            ELSE 'sin especificar'
        END as tipo_orientacion
FROM    propiedades.silver.propiedades
ORDER BY orientacion
;

## 1.2: Crear tabla de hechos (fact_propiedades)


In [0]:
%sql
-- Crea Fact: fact_propiedades

DROP TABLE IF EXISTS propiedades.gold.fact_propiedades;

CREATE TABLE propiedades.gold.fact_propiedades
(
    row_hash STRING NOT NULL,
    zona_id BIGINT,
    tipo_operacion_id BIGINT,
    fecha_id BIGINT,
    caracteristicas_id BIGINT,
    orientacion_id BIGINT,
    precio DECIMAL(15,2),
    expensas DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    metros_cuadrados_totales DECIMAL(15,2),
    metros_cuadrados_cubiertos DECIMAL(15,2),
    ambientes INT,
    url STRING,
    _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES
(
     'delta.feature.allowColumnDefaults' = 'supported'
)
COMMENT  'Tabla de hechos de propiedades - Star Schema. PK = row_hash (MD5 de url + precio)';

-----------------------------------------------------------------
-- INSERTA FACT
-----------------------------------------------------------------

INSERT INTO propiedades.gold.fact_propiedades
(   row_hash, zona_id, tipo_operacion_id, fecha_id, caracteristicas_id, orientacion_id,
    precio, expensas, precio_por_m2,
    metros_cuadrados_totales, metros_cuadrados_cubiertos, ambientes, url
)
SELECT  
        MD5(concat_ws('|', sp.url, cast(sp.precio as string ))) as row_hash,
        dz.zona_id,
        dt.tipo_operacion_id,
        dtf.fecha_id,
        dc.caracteristicas_id,
        do.orientacion_id,
        sp.precio, sp.expensas, sp.precio_por_m2,
        sp.metros_cuadrados_totales, sp.metros_cuadrados_cubiertos,
        sp.ambientes, sp.url

FROM    propiedades.silver.propiedades sp
LEFT JOIN propiedades.gold.dim_zona dz 
    ON sp.partido = dz.partido AND sp.region = dz.region
LEFT JOIN propiedades.gold.dim_tipo_operacion dt 
    ON sp.tipo_operacion = dt.tipo_operacion AND sp.moneda = dt.moneda
LEFT JOIN propiedades.gold.dim_tiempo dtf 
    ON sp.fecha_publicacion = dtf.fecha
LEFT JOIN propiedades.gold.dim_caracteristicas dc 
    ON COALESCE(sp.estado, 'sin especificar') = dc.estado AND COALESCE(sp.cochera, false) = dc.cochera
LEFT JOIN propiedades.gold.dim_orientacion do 
    ON COALESCE(sp.orientacion, 'sin especificar') = do.orientacion
LIMIT 10000;

